# Instalar dependencias

In [1]:
pip install tensorflow-cpu streamlit matplotlib seaborn scikit-learn pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.3/274.3 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 11.4 MB/s eta 0:00:00


Verificar que el entorno este corriendo con la GPU

In [2]:
import tensorflow as tf
print("Versión de TensorFlow:", tf.__version__)
dispositivos_gpu = tf.config.list_physical_devices('GPU')
if dispositivos_gpu:
    print(f"✅ ¡Éxito! Conectado a la GPU: {dispositivos_gpu}")
else:
    print("❌ ERROR: Sigues usando la CPU. Ve a Entorno de ejecución -> Cambiar tipo de entorno de ejecución y selecciona 'T4 GPU'.")


Versión de TensorFlow: 2.21.0
✅ ¡Éxito! Conectado a la GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# Entrenar el módelo

In [3]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models, mixed_precision

# 1. Mantener la velocidad de la GPU
mixed_precision.set_global_policy('mixed_float16')

# 2. Cargar el dataset COMPLETO (50,000 imágenes de entrenamiento)
(X_train_full, y_train_full), (X_test, y_test) = datasets.cifar10.load_data()

# Normalizar todas las imágenes
X_train_full = X_train_full.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Separar 5,000 para validación y usar 45,000 para entrenar
X_val = X_train_full[-5000:]
y_val = y_train_full[-5000:]
X_train = X_train_full[:-5000]
y_train = y_train_full[:-5000]

# 3. Capas de Data Augmentation integradas
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# 4. Arquitectura Avanzada (Más profunda, con Dropout y BatchNormalization)
model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation, # Modifica las fotos al vuelo para que el modelo no se las memorice

    # Bloque Convolucional 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25), # Apaga neuronas al azar para forzar un aprendizaje robusto

    # Bloque Convolucional 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    # Clasificador Densa
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax', dtype='float32') # float32 por seguridad en precisión mixta
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 5. Entrenamiento a fondo (20 Épocas)
print("\nIniciando entrenamiento de Alta Precisión...")
model.fit(X_train, y_train,
          epochs=20,
          validation_data=(X_val, y_val),
          batch_size=256) # Batch size optimizado para balancear velocidad y aprendizaje

# 6. Guardar el nuevo modelo mejorado
model.save('modelo_cifar10.keras')
print("¡Modelo de alta precisión guardado exitosamente!")


Iniciando entrenamiento de Alta Precisión...
Epoch 1/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 17s 46ms/step - accuracy: 0.3837 - loss: 1.8565 - val_accuracy: 0.0970 - val_loss: 3.7299
Epoch 2/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 14s 33ms/step - accuracy: 0.5017 - loss: 1.3935 - val_accuracy: 0.2046 - val_loss: 2.7983
Epoch 3/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 6s 35ms/step - accuracy: 0.5504 - loss: 1.2676 - val_accuracy: 0.5942 - val_loss: 1.1481
Epoch 4/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.5809 - loss: 1.1813 - val_accuracy: 0.6102 - val_loss: 1.1177
Epoch 5/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 6s 35ms/step - accuracy: 0.6025 - loss: 1.1283 - val_accuracy: 0.6156 - val_loss: 1.0859
Epoch 6/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.6229 - loss: 1.0735 - val_accuracy: 0.5868 - val_loss: 1.2466
Epoch 7/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 6s 35ms/step - accuracy: 0.6336 - loss: 1.0360 - val_accuracy: 0.6298 - val_loss: 1.0383
Epoch 8/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/

# Descargar módelo entrenado

In [4]:
from google.colab import files
files.download('modelo_cifar10.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>